In [11]:
!pip install -q gspread faker

In [12]:
# Importa as ferramentas para autenticação, geração de dados e sorteios.
import json, gspread
from google.colab import userdata
from faker import Faker
import random

# Lê a credencial salva nos Secrets e conecta à planilha.
info = json.loads(userdata.get("GCP_SA_JSON"))
gc = gspread.service_account_from_dict(info)
# Exibe o e-mail da conta autenticada.
print("Conectado como:", info["client_email"])

Conectado como: desafio1-python@neural-vista-513819-u4.iam.gserviceaccount.com


In [13]:
# Informe o ID da planilha e o nome que será dado à aba.
SHEET_ID = "1h2z1NeBQKo4zktvMk7v8drEZCE31zLSTPvoLyi-850U"
NOME_TABELA = "desafio_01_festival_musica"   # nome da tabela na ficha do tema, ex.: "vagas"

# Abre a planilha e seleciona sua primeira aba.
sh = gc.open_by_key(SHEET_ID)
ws = sh.sheet1
# Renomeia a aba e exibe os dados da planilha.
ws.update_title(NOME_TABELA)     # a aba passa a ter o nome da tabela
print(sh.title, "|", ws.title, "|", sh.url)

Desafio_01_Festival_Musica | desafio_01_festival_musica | https://docs.google.com/spreadsheets/d/1h2z1NeBQKo4zktvMk7v8drEZCE31zLSTPvoLyi-850U


In [14]:
# Substitua pelos nomes EXATOS de coluna da ficha do seu tema
# Define as colunas que formarão o cabeçalho da tabela.
COLUNAS = ["atracao_id", "nome_artista_ficticio", "genero_musical", "edicao_festival", "dia_apresentacao", "palco", "horario_inicio"]

# Escreve o cabeçalho na primeira célula e confere o resultado.
ws.update(range_name="A1", values=[COLUNAS])
print("Cabeçalho confere:", ws.row_values(1) == COLUNAS)

Cabeçalho confere: True


In [15]:
# Configura os geradores aleatórios para produzir resultados reproduzíveis.
fake = Faker("pt_BR")
Faker.seed(42)
random.seed(42)

def gerar_linha(i):
  edicao = random.choice(EDICOES)
    # Cria um registro fictício para as colunas definidas no esquema
    # Personalize de acordo com o seu esquema de dados
    return {
        "atracao_id": i,
        "nome_artista_ficticio": random.choice(["NOME_ARTISTAS_FICTICIO"]),
        "genero_musical": random.choice(["GENEROS_MUSICAIS"]),
        "edicao_festival": edicao(["edicao"]),
        "dia_apresentacao": random.choice(edicao["dias"]),
        "palco": random.choice(["PALCOS"]),
        "horario_inicio": random.choice(["HORARIOS_INICIO"]),
    }

# Gera 10 registros numerados de 1 a 10.
linhas = [gerar_linha(i) for i in range(1, 11)]

# Organiza os valores na ordem do cabeçalho; nome de coluna incorreto gera KeyError.
valores = [[linha[c] for c in COLUNAS] for linha in linhas]
# Acrescenta os registros à planilha e informa quantos foram enviados.
ws.append_rows(valores, value_input_option="RAW")
print(len(valores), "linhas enviadas")

10 linhas enviadas


In [16]:
# Lê todos os valores da aba para conferir o conteúdo enviado.
valores_lidos = ws.get_all_values()
# Exibe a quantidade de linhas, confere o cabeçalho e mostra exemplos.
print("Linhas (cabeçalho + dados):", len(valores_lidos))    # esperado: 11
print("Cabeçalho confere:", valores_lidos[0] == COLUNAS)
print(valores_lidos[:3])

Linhas (cabeçalho + dados): 31
Cabeçalho confere: True
[['atracao_id', 'nome_artista_ficticio', 'genero_musical', 'edicao_festival', 'dia_apresentacao', 'palco', 'horario_inicio'], ['1', 'Brenda Alves', 'Rock', '1ª Edição', 'Domingo', 'Palco Indie', '16:00'], ['2', 'Sra. Isabelly Câmara', 'Pop', '1ª Edição', 'Domingo', 'Palco Principal', '22:00']]


In [17]:
# Importa recursos para controlar pausas, acessar a planilha e tratar tabelas.
import time
import gspread
import pandas as pd

def df_para_valores(df):
    """DataFrame -> lista de listas com tipos nativos do Python."""
    # Trabalha em uma cópia para não alterar o DataFrame original.
    df = df.copy()
    for c in df.columns:
        # Converte datas para texto em um formato padronizado.
        if pd.api.types.is_datetime64_any_dtype(df[c]):
            df[c] = df[c].dt.strftime("%Y-%m-%d %H:%M:%S")
        # Converte outros valores de data e hora para texto.
        elif df[c].dtype == object:
            df[c] = df[c].map(
                lambda v: v.isoformat() if hasattr(v, "isoformat") else v)
    # Substitui valores ausentes por texto vazio e retorna linhas de valores.
    return df.astype(object).where(df.notna(), "").values.tolist()

def enviar(sh, df, nome_aba, lote=5000, pausa=2):
    """Escreve cabeçalho e linhas numa aba, em lotes, e confere a contagem."""
    # Formata números com ponto como separador de milhar.
    n = lambda x: f"{x:,}".replace(",", ".")
    try:
        # Reutiliza a aba existente e apaga seu conteúdo anterior.
        ws = sh.worksheet(nome_aba)
        ws.clear()
    except gspread.WorksheetNotFound:
        # Cria uma aba caso ela ainda não exista.
        ws = sh.add_worksheet(title=nome_aba, rows=len(df) + 1,
                              cols=len(df.columns))
    # Ajusta o tamanho da aba e grava os nomes das colunas.
    ws.resize(rows=len(df) + 1, cols=len(df.columns))
    ws.update(range_name="A1", values=[list(df.columns)])
    # Converte o DataFrame e envia as linhas em lotes.
    valores = df_para_valores(df)
    for i in range(0, len(valores), lote):
        ws.append_rows(valores[i:i + lote], value_input_option="RAW")
        print(f"  {nome_aba}: {n(min(i + lote, len(valores)))} / {n(len(valores))}")
        time.sleep(pausa)               # respeita o limite de escritas/minuto
    # Confere se a quantidade de linhas gravadas corresponde à esperada.
    lidas = len(ws.col_values(1)) - 1   # lê de volta e confere
    if lidas != len(df):
        raise RuntimeError(f"Aba {nome_aba}: enviei {len(df)}, li {lidas}.")
    print(f"OK {nome_aba}: {n(len(df))} linhas na planilha")

def remover_aba_padrao(sh):
    """Apaga a aba vazia que o Google cria junto com a planilha nova."""
    # Localiza e remove a aba padrão somente se houver outra aba disponível.
    abas = sh.worksheets()
    padrao = ("Página1", "Sheet1", "Planilha1")
    for w in abas:
        if w.title in padrao and len(abas) > 1 and not w.get_all_values():
            sh.del_worksheet(w)

In [19]:
# ==========================================================
# CÉLULA 8 - CARREGAR BASES DE DADOS
# PROJETO: 08 - FESTIVAL DE MÚSICA
# ==========================================================

import os
import pandas as pd
from google.colab import files

# 1. Define a pasta do projeto
PASTA_PROJETO = "08-festival-de-musica"

# Cria a pasta caso não exista
os.makedirs(PASTA_PROJETO, exist_ok=True)

print("PROJETO: FESTIVAL DE MÚSICA")
print("=" * 50)
print("Selecione os dois arquivos CSV:")
print("1. atracoes.csv")
print("2. ingressos_vendidos.csv")
print("=" * 50)

# 2. Faz o upload dos arquivos
arquivos = files.upload()

# 3. Identifica os arquivos e padroniza os nomes
arquivos_encontrados = {}

for nome, conteudo in arquivos.items():

    nome_base = os.path.basename(nome).lower()

    if nome_base.startswith("atracoes") and nome_base.endswith(".csv"):
        nome_correto = "atracoes.csv"

    elif (
        nome_base.startswith("ingressos_vendidos")
        and nome_base.endswith(".csv")
    ):
        nome_correto = "ingressos_vendidos.csv"

    else:
        print(f"Arquivo ignorado: {nome}")
        continue

    caminho = os.path.join(PASTA_PROJETO, nome_correto)

    with open(caminho, "wb") as arquivo:
        arquivo.write(conteudo)

    arquivos_encontrados[nome_correto] = caminho

    print(f"OK! Arquivo salvo: {caminho}")

# 4. Verifica se os arquivos existem
arquivos_necessarios = [
    "atracoes.csv",
    "ingressos_vendidos.csv"
]

for nome in arquivos_necessarios:

    if nome not in arquivos_encontrados:
        raise FileNotFoundError(
            f"Arquivo não encontrado no upload: {nome}"
        )

# 5. Carrega as tabelas
BASES = {
    "atracoes": pd.read_csv(
        arquivos_encontrados["atracoes.csv"]
    ),

    "ingressos_vendidos": pd.read_csv(
        arquivos_encontrados["ingressos_vendidos.csv"]
    )
}

# 6. Calcula a quantidade total de registros
total = sum(len(df) for df in BASES.values())

print("\nRESUMO DAS BASES CARREGADAS")
print("=" * 50)

for nome, df in BASES.items():

    print(f"\nTabela: {nome}")
    print(f"Quantidade de linhas: {len(df):,}")
    print(f"Quantidade de colunas: {len(df.columns)}")
    print(f"Colunas: {list(df.columns)}")

print("\n" + "=" * 50)
print(f"TOTAL DE REGISTROS: {total:,}")

# 7. Verifica o requisito de 20.000 registros
if total >= 20000:
    print("\nOK! Volume mínimo de 20.000 registros atingido.")
else:
    print("\nATENÇÃO: Volume abaixo do mínimo exigido.")
    print(f"Faltam {20000 - total:,} registros.")

# 8. Confere se os IDs das vendas existem nas atrações
if (
    "atracao_id" in BASES["atracoes"].columns
    and "atracao_id" in BASES["ingressos_vendidos"].columns
):
    ids_atracoes = set(BASES["atracoes"]["atracao_id"])
    ids_vendas = set(BASES["ingressos_vendidos"]["atracao_id"])

    ids_invalidos = ids_vendas - ids_atracoes

    if not ids_invalidos:
        print("\nOK! Todos os IDs das vendas existem em atracoes.")
    else:
        print("\nATENÇÃO! Existem IDs sem atração correspondente.")
        print(f"IDs inválidos: {list(ids_invalidos)[:10]}")
else:
    print("\nVerifique as colunas de relacionamento entre as tabelas.")

# 9. Exibe amostras das tabelas
print("\nAMOSTRA - TABELA ATRACOES")
display(BASES["atracoes"].head())

print("\nAMOSTRA - TABELA INGRESSOS VENDIDOS")
display(BASES["ingressos_vendidos"].head())

print("\nCÉLULA 8 EXECUTADA COM SUCESSO!")

PROJETO: FESTIVAL DE MÚSICA
Selecione os dois arquivos CSV:
1. atracoes.csv
2. ingressos_vendidos.csv


Saving atracoes.csv to atracoes (3).csv
Saving ingressos_vendidos.csv to ingressos_vendidos (4).csv
OK! Arquivo salvo: 08-festival-de-musica/atracoes.csv
OK! Arquivo salvo: 08-festival-de-musica/ingressos_vendidos.csv

RESUMO DAS BASES CARREGADAS

Tabela: atracoes
Quantidade de linhas: 10,100
Quantidade de colunas: 7
Colunas: ['atracao_id', 'nome_artista_ficticio', 'genero_musical', 'edicao_festival', 'dia_apresentacao', 'palco', 'horario_inicio']

Tabela: ingressos_vendidos
Quantidade de linhas: 55,000
Quantidade de colunas: 7
Colunas: ['venda_id', 'atracao_id', 'tipo_ingresso', 'canal_venda', 'data_hora_venda', 'valor', 'forma_pagamento']

TOTAL DE REGISTROS: 65,100

OK! Volume mínimo de 20.000 registros atingido.

OK! Todos os IDs das vendas existem em atracoes.

AMOSTRA - TABELA ATRACOES


,atracao_id,nome_artista_ficticio,genero_musical,edicao_festival,dia_apresentacao,palco,horario_inicio
0,1,Banda Moura,sertanejo,Festival Illo 2025,2025-01-23,eletronico,20:45
1,2,Coletivo Casa Grande,rap,Festival Nobis 2026,2026-04-20,secundario,13:45
2,3,Banda da Rosa,eletronica,Festival Voluptate 2023,2023-05-23,secundario,12:00
3,4,Banda Farias,rap,Festival Iusto 2026,2026-10-11,principal,16:00
4,5,Rodrigues,reggae,Festival Aperiam 2023,2023-04-06,eletronico,12:30



AMOSTRA - TABELA INGRESSOS VENDIDOS


,venda_id,atracao_id,tipo_ingresso,canal_venda,data_hora_venda,valor,forma_pagamento
0,1,3291,pista,bilheteria_fisica,2022-11-12 17:19:45,122.58,pix
1,2,66,pista_premium,app,2026-04-27 13:42:21,429.29,cartao_credito
2,3,8369,pista,site,2022-11-01 14:04:35,112.52,pix
3,4,8497,pista_premium,site,2021-04-16 12:50:44,459.93,cartao_credito
4,5,9073,pista,app,2020-01-24 15:19:11,190.61,pix



CÉLULA 8 EXECUTADA COM SUCESSO!


In [20]:
# Envia cada tabela para uma aba e remove a aba padrão, se estiver vazia.
for nome, df in BASES.items():
    enviar(sh, df, nome)        # uma aba por tabela
remover_aba_padrao(sh)

  atracoes: 5.000 / 10.100
  atracoes: 10.000 / 10.100
  atracoes: 10.100 / 10.100
OK atracoes: 10.100 linhas na planilha
  ingressos_vendidos: 5.000 / 55.000
  ingressos_vendidos: 10.000 / 55.000
  ingressos_vendidos: 15.000 / 55.000
  ingressos_vendidos: 20.000 / 55.000
  ingressos_vendidos: 25.000 / 55.000
  ingressos_vendidos: 30.000 / 55.000
  ingressos_vendidos: 35.000 / 55.000
  ingressos_vendidos: 40.000 / 55.000
  ingressos_vendidos: 45.000 / 55.000
  ingressos_vendidos: 50.000 / 55.000
  ingressos_vendidos: 55.000 / 55.000
OK ingressos_vendidos: 55.000 linhas na planilha


In [21]:
# Soma e exibe a quantidade total de linhas das tabelas.
total = sum(len(d) for d in BASES.values())
print("Total de linhas:", f"{total:,}".replace(",", "."))
# Interrompe a execução se o volume mínimo não for atingido.
assert total >= 20_000, "Abaixo do mínimo (20.000 linhas somadas)."

# Confere os cabeçalhos e a quantidade de registros em cada aba.
for nome, df in BASES.items():
    w = sh.worksheet(nome)
    assert w.row_values(1) == list(df.columns), f"Cabeçalho diferente em {nome}"
    assert len(w.col_values(1)) - 1 == len(df), f"Contagem diferente em {nome}"
# Confirma que todas as verificações passaram.
print("A planilha confere com a base gerada.")

Total de linhas: 65.100
A planilha confere com a base gerada.
